# E8: vit_large_patch16_dinov3.lvd1689m with NoFusion Fusion:

### Experiment identity:
- Experiment ID: `E8`
- Reference script: `experiments/ablation/E8_NoFusion_Meta_cv.py`
- Commit: `fc6f5e3`
- SHA256: `52042871a6522261c524744df2015a005cdb3f21901d5045656bf416c7d2cb6f`
- Backbone model: `vit_large_patch16_dinov3.lvd1689m`
- Fusion block: `NoFusion`
- Metadata state: `True`
- Output head: Compositional Softplus (Green, Dead, Clover -> GDM, Total)



### Source parity notice:
Copied standalone from `experiments/ablation/E8_NoFusion_Meta_cv.py`, `src/engine.py`, and `src/models.py`. Contains complete self-contained execution logic without repository imports.


## 2. Protocol and scientific purpose:

Evaluates no cross-view fusion block combined with 23 tabular metadata features.


## 3. Requirements and expected resources:

- Hardware: GPU recommended (NVIDIA RTX 4060 Laptop or Kaggle T4/P100).
- Batch size: 4, Image size: 448x448.
- Expected memory: ~6-8 GB VRAM.
- Packages: `torch`, `torchvision`, `timm`, `albumentations`, `opencv-python`, `pandas`.


In [ ]:
# 4. Configuration cell:
RUN_FULL = False

class CFG:
    SEED = 17
    N_FOLDS = 5
    FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
    DATA_DIR = 'csiro-biomass'
    TRAIN_CSV = 'csiro-biomass/train.csv'
    TRAIN_IMAGE_DIR = 'csiro-biomass/train'
    FOLD_FILE = 'output/reruns_2026_09_13/folds_seed17.csv'
    OUTPUT_DIR = 'output/reruns_2026_09_13/E8_E8_NoFusion_Meta'
    MODEL_NAME = 'vit_large_patch16_dinov3.lvd1689m'
    IMG_SIZE = 448
    BATCH_SIZE = 4
    NUM_WORKERS = 2
    EPOCHS = 50
    WARMUP_EPOCHS = 5
    LR_BACKBONE = 1e-05
    LR_HEAD = 0.0005
    WD = 1e-2
    CLIP_GRAD_NORM = 1.0
    DROPOUT = 0.2
    HUBER_BETA = 5.0
    USE_METADATA = True
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    TARGET_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]


In [ ]:
# 5. Environment and seed setup:
import os
import sys
import gc
import math
import random
import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedGroupKFold, KFold
import timm

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 17) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

print("Environment information:")
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))
print("timm version:", timm.__version__)
print("albumentations version:", A.__version__)
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)


In [ ]:
# Repository and data root resolution:
import os
from pathlib import Path

def resolve_data_and_repo_roots() -> Tuple[Path, Path]:
    repo_candidates = [
        Path(os.environ.get("REPO_ROOT", "")),
        Path(".").resolve(),
        Path("..").resolve(),
        Path("../..").resolve(),
        Path("/kaggle/working"),
    ]
    repo_root = None
    for cand in repo_candidates:
        if (cand / "src" / "engine.py").is_file() and (cand / "experiments").is_dir():
            repo_root = cand
            break
    if repo_root is None:
        repo_root = Path(".").resolve()

    data_candidates = [
        Path(os.environ.get("BIOMASS_DATA_DIR", "")),
        repo_root / "csiro-biomass",
        repo_root.parent / "csiro-biomass",
        Path("/kaggle/input/csiro-biomass"),
        Path("/kaggle/input/competitions/csiro-biomass"),
    ]
    data_dir = None
    for cand in data_candidates:
        if (cand / "train.csv").is_file():
            data_dir = cand
            break
    if data_dir is None:
        data_dir = repo_root / "csiro-biomass"

    return repo_root, data_dir

# 6. Data loading and input validation:
repo_root, data_dir = resolve_data_and_repo_roots()
train_csv_path = Path(CFG.TRAIN_CSV) if Path(CFG.TRAIN_CSV).is_file() else (data_dir / "train.csv")

if not train_csv_path.is_file():
    raise FileNotFoundError(
        f"train.csv not found at {train_csv_path}. Please check data path configuration."
    )

print(f"Loading data from: {train_csv_path}")
df_long = pd.read_csv(train_csv_path)

if "sample_id" not in df_long.columns:
    raise ValueError(f"train.csv missing sample_id column: {df_long.columns.tolist()}")

df_long["image_id"] = df_long["sample_id"].str.split("__").str[0]

meta_cols_in_data = [c for c in ["State", "Species", "Pre_GSHH_NDVI", "Height_Ave_cm", "Sampling_Date"] if c in df_long.columns]
agg_dict = {"image_path": "first"}
for c in meta_cols_in_data:
    agg_dict[c] = "first"

df_wide = df_long.pivot_table(
    index=["image_id"],
    columns="target_name",
    values="target",
    aggfunc="first"
).reset_index()

df_meta = df_long.groupby("image_id").agg(agg_dict).reset_index()
df_wide = pd.merge(df_wide, df_meta, on="image_id", how="left")

for col in CFG.TARGET_COLS:
    if col not in df_wide.columns:
        df_wide[col] = 0.0

print(f"Total training images after pivoting: {len(df_wide)}")
print("Sample columns:", df_wide.columns.tolist()[:8])


In [ ]:
# 7. Fold construction or locked fold loading:
fold_file_path = Path(CFG.FOLD_FILE)
if fold_file_path.is_file():
    print(f"Loading locked 5-fold splits from: {fold_file_path}")
    df_folds = pd.read_csv(fold_file_path)
    df_wide = pd.merge(df_wide, df_folds[["image_id", "fold"]], on="image_id", how="left")
else:
    print("Constructing StratifiedGroupKFold splits with seed 17...")
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    total_bins = pd.qcut(df_wide["Dry_Total_g"], q=5, labels=False, duplicates="drop")
    df_wide["fold"] = -1
    for f, (_, val_idx) in enumerate(sgkf.split(df_wide, total_bins, groups=df_wide["image_id"])):
        df_wide.loc[val_idx, "fold"] = f

print("Fold distribution:")
print(df_wide["fold"].value_counts().sort_index())


In [ ]:
# 8. Transforms and dataset classes:
def get_transforms(img_size: int):
    train_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    val_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    return train_transform, val_transform

class BiomassDualViewDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir: Path, transform=None, meta_array: Optional[np.ndarray] = None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.meta_array = meta_array
        self.target_cols = CFG.TARGET_COLS

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image_path_rel = row["image_path"]
        img_name = os.path.basename(image_path_rel)
        full_img_path = self.image_dir / img_name
        if not full_img_path.is_file():
            full_img_path = self.image_dir / image_path_rel

        img = cv2.imread(str(full_img_path))
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape
        mid = w // 2
        left_img = img[:, :mid]
        right_img = img[:, mid:]

        if self.transform is not None:
            left_t = self.transform(image=left_img)["image"]
            right_t = self.transform(image=right_img)["image"]
        else:
            left_t = ToTensorV2()(image=left_img)["image"]
            right_t = ToTensorV2()(image=right_img)["image"]

        targets = row[self.target_cols].to_numpy().astype(np.float32)

        item = {
            "image_id": row["image_id"],
            "left": left_t,
            "right": right_t,
            "targets": torch.tensor(targets, dtype=torch.float32),
        }
        if self.meta_array is not None:
            item["metadata"] = torch.tensor(self.meta_array[idx], dtype=torch.float32)
        return item

# Metadata encoding (23 dimensions):
def encode_metadata(df: pd.DataFrame) -> Tuple[np.ndarray, List[str]]:
    states = ["NSW", "QLD", "TAS", "VIC"]
    species_list = [
        "Brachiaria decumbens", "Chloris gayana", "Digitaria eriantha",
        "Festuca arundinacea", "Lolium multiflorum", "Lolium perenne",
        "Megathyrsus maximus", "Mixed", "Paspalum dilatatum",
        "Pennisetum clandestinum", "Setaria sphacelata",
        "Trifolium repens/Lolium perenne", "Trifolium subterraneum",
        "Trifolium subterraneum/Lolium perenne",
        "Trifolium subterraneum/Phalaris aquatica"
    ]
    encoded = []
    names = []

    # One hot State:
    for s in states:
        encoded.append((df["State"] == s).astype(np.float32).to_numpy()[:, None])
        names.append(f"State_{s}")

    # One hot Species:
    for sp in species_list:
        encoded.append((df["Species"] == sp).astype(np.float32).to_numpy()[:, None])
        names.append(f"Species_{sp}")

    # Continuous NDVI and Height:
    ndvi = df["Pre_GSHH_NDVI"].fillna(0.0).astype(np.float32).to_numpy()[:, None]
    height = (df["Height_Ave_cm"].fillna(0.0) / 100.0).astype(np.float32).to_numpy()[:, None]
    encoded.extend([ndvi, height])
    names.extend(["Pre_GSHH_NDVI", "Height_m"])

    # Cyclical Month:
    months = pd.to_datetime(df["Sampling_Date"]).dt.month.fillna(1).to_numpy()
    sin_m = np.sin(2 * np.pi * months / 12.0).astype(np.float32)[:, None]
    cos_m = np.cos(2 * np.pi * months / 12.0).astype(np.float32)[:, None]
    encoded.extend([sin_m, cos_m])
    names.extend(["Month_sin", "Month_cos"])

    mat = np.hstack(encoded).astype(np.float32)
    return mat, names


In [ ]:
# 9. Model and fusion definitions:
# Identity / No-Fusion Module:
def get_fusion_module(dim: int, dropout: float):
    return nn.Identity()


def _make_head(nf: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(nf, nf // 2),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(nf // 2, 1),
        nn.Softplus()
    )

class StandaloneBiomassModel(nn.Module):
    def __init__(self, model_name: str, img_size: int, dropout: float = 0.2, use_metadata: bool = True, meta_dim: int = 23):
        super().__init__()
        backbone_kwargs = dict(pretrained=True, num_classes=0, global_pool='')
        if img_size is not None and model_name.startswith('vit_'):
            backbone_kwargs['img_size'] = img_size
        self.backbone = timm.create_model(model_name, **backbone_kwargs)
        if hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(True)
        nf = self.backbone.num_features
        self.fusion = get_fusion_module(nf, dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.use_metadata = use_metadata
        if use_metadata:
            meta_hidden = 64
            self.meta_mlp = nn.Sequential(
                nn.Linear(meta_dim, meta_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(meta_hidden, meta_hidden),
            )
            self.meta_proj = nn.Sequential(
                nn.Linear(nf + meta_hidden, nf),
                nn.GELU(),
            )

        self.head_green = _make_head(nf, dropout)
        self.head_dead = _make_head(nf, dropout)
        self.head_clover = _make_head(nf, dropout)

    def _as_tokens(self, x):
        if x.ndim == 4:
            return x.flatten(2).transpose(1, 2)
        if x.ndim == 2:
            return x.unsqueeze(1)
        return x

    def forward(self, left, right, metadata=None):
        x_l = self._as_tokens(self.backbone(left))
        x_r = self._as_tokens(self.backbone(right))
        x = torch.cat([x_l, x_r], dim=1)
        x = self.fusion(x)
        x = self.pool(x.transpose(1, 2)).flatten(1)

        if self.use_metadata and metadata is not None:
            m_feat = self.meta_mlp(metadata)
            x = self.meta_proj(torch.cat([x, m_feat], dim=1))

        green = self.head_green(x)
        dead = self.head_dead(x)
        clover = self.head_clover(x)
        gdm = green + clover
        total = gdm + dead
        return torch.cat([green, dead, clover, gdm, total], dim=1)

def build_model():
    return StandaloneBiomassModel(CFG.MODEL_NAME, CFG.IMG_SIZE, CFG.DROPOUT, CFG.USE_METADATA)


In [ ]:
# 10. Loss and metric definitions:
def compute_weighted_r2(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, np.ndarray]:
    weights = np.array(CFG.TARGET_WEIGHTS, dtype=np.float64)
    scores = []
    for i in range(y_true.shape[1]):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        ss_res = np.sum((y_t - y_p) ** 2)
        ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
        scores.append(r2)
    per_target = np.array(scores, dtype=np.float64)
    weighted = float(np.sum(per_target * weights))
    return weighted, per_target

class CompositionalHuberLoss(nn.Module):
    def __init__(self, beta: float = 5.0, weights: Optional[List[float]] = None):
        super().__init__()
        self.beta = beta
        self.weights = torch.tensor(weights if weights is not None else CFG.TARGET_WEIGHTS, dtype=torch.float32)

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        diff = torch.abs(y_pred - y_true)
        huber = torch.where(diff < self.beta, 0.5 * (diff ** 2) / self.beta, diff - 0.5 * self.beta)
        w = self.weights.to(y_pred.device)
        weighted_loss = huber * w.unsqueeze(0)
        return torch.mean(torch.sum(weighted_loss, dim=-1))


In [ ]:
# 11. Training and validation functions:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        left = batch["left"].to(device)
        right = batch["right"].to(device)
        targets = batch["targets"].to(device)
        meta = batch["metadata"].to(device) if "metadata" in batch else None

        optimizer.zero_grad()
        with autocast("cuda", enabled=torch.cuda.is_available()):
            outputs = model(left, right, meta) if meta is not None else model(left, right)
            loss = criterion(outputs, targets)

        if scaler is not None and torch.cuda.is_available():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_GRAD_NORM)
            optimizer.step()

        total_loss += loss.item() * len(targets)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def validate_epoch(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    for batch in loader:
        left = batch["left"].to(device)
        right = batch["right"].to(device)
        targets = batch["targets"].numpy()
        meta = batch["metadata"].to(device) if "metadata" in batch else None

        with autocast("cuda", enabled=torch.cuda.is_available()):
            outputs = model(left, right, meta) if meta is not None else model(left, right)

        all_preds.append(outputs.cpu().numpy())
        all_targets.append(targets)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    weighted_r2, per_target_r2 = compute_weighted_r2(y_true, y_pred)
    return weighted_r2, per_target_r2, y_pred


In [ ]:
# 12. Five fold execution:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

out_dir = Path(CFG.OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

if not RUN_FULL:
    print("Safe mode active (RUN_FULL = False): skipping full multi-epoch training.")
    print(f"Configuration verified: {CFG.N_FOLDS} folds, image size {CFG.IMG_SIZE}, model {CFG.MODEL_NAME}.")
    print("Set RUN_FULL = True to execute full cross-validation.")
else:
    print("Starting full 5-fold cross-validation...")
    train_transform, val_transform = get_transforms(CFG.IMG_SIZE)
    meta_array, meta_names = encode_metadata(df_wide) if CFG.USE_METADATA else (None, [])
    
    oof_predictions = np.zeros((len(df_wide), len(CFG.TARGET_COLS)), dtype=np.float32)
    fold_scores = []

    for fold in CFG.FOLDS_TO_TRAIN:
        print(f"\n--- Fold {fold} ---")
        train_mask = df_wide["fold"] != fold
        val_mask = df_wide["fold"] == fold

        train_ds = BiomassDualViewDataset(
            df_wide[train_mask],
            Path(CFG.TRAIN_IMAGE_DIR),
            transform=train_transform,
            meta_array=meta_array[train_mask] if meta_array is not None else None
        )
        val_ds = BiomassDualViewDataset(
            df_wide[val_mask],
            Path(CFG.TRAIN_IMAGE_DIR),
            transform=val_transform,
            meta_array=meta_array[val_mask] if meta_array is not None else None
        )

        train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

        model = build_model().to(device)
        optimizer = optim.AdamW([
            {"params": [p for n, p in model.named_parameters() if "backbone" in n], "lr": CFG.LR_BACKBONE},
            {"params": [p for n, p in model.named_parameters() if "backbone" not in n], "lr": CFG.LR_HEAD},
        ], weight_decay=CFG.WD)

        criterion = CompositionalHuberLoss(beta=CFG.HUBER_BETA)
        scaler = GradScaler("cuda") if torch.cuda.is_available() else None

        best_score = -float("inf")
        best_preds = None

        for epoch in range(CFG.EPOCHS):
            loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
            val_r2, val_per_target, val_preds = validate_epoch(model, val_loader, device)
            if val_r2 > best_score:
                best_score = val_r2
                best_preds = val_preds
                ckpt_path = out_dir / f"fold_{fold}_best.pth"
                torch.save(model.state_dict(), ckpt_path)

        print(f"Fold {fold} Best Weighted R2: {best_score:.4f}")
        oof_predictions[val_mask] = best_preds
        fold_scores.append(best_score)


In [ ]:
# 13. OOF and aggregate evaluation:
if not RUN_FULL:
    print("Safe mode active: OOF and aggregate evaluation skipped.")
else:
    y_true_all = df_wide[CFG.TARGET_COLS].to_numpy().astype(np.float32)
    overall_r2, per_target_r2 = compute_weighted_r2(y_true_all, oof_predictions)
    print(f"\n=== Overall 5-Fold Pooled OOF Weighted R2: {overall_r2:.4f} ===")
    for col, score in zip(CFG.TARGET_COLS, per_target_r2):
        print(f"  {col}: {score:.4f}")
    print(f"Mean Fold Score: {np.mean(fold_scores):.4f} (dispersion ddof=0: {np.std(fold_scores):.4f})")


In [ ]:
# 14. Artifact manifest and run metadata:
manifest = {
    "model_name": CFG.MODEL_NAME,
    "seed": CFG.SEED,
    "folds": CFG.N_FOLDS,
    "image_size": CFG.IMG_SIZE,
    "use_metadata": CFG.USE_METADATA,
    "output_dir": str(CFG.OUTPUT_DIR),
    "git_commit": "fc6f5e3",
    "run_full": RUN_FULL,
}

manifest_path = Path(CFG.OUTPUT_DIR) / "run_metadata.json"
if RUN_FULL:
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"Run metadata written to: {manifest_path}")
else:
    print("Safe mode active: planned manifest would be written to:", manifest_path)
    print("Manifest content:")
    print(json.dumps(manifest, indent=2))


## 15. Limits and interpretation:

- Single seed limitation: cross-validation evaluates fold dispersion for seed 17, not stochastic variance across multiple initializations.
- Resource constraints: mixed precision (AMP) is enabled. High-resolution fine-tuning may require gradient accumulation on smaller GPUs.
